# Incremental Loading of CSV Data into Bronze Flights Table

Efficiently extract rows from the CSV file and incrementally load the data into the bronze_flights table for scalable processing.
The load is performed incrementally, where if any new file is uploaded and the job is run the file will be identified and the contents will be loaded into the table.
* Make sure not to include the headers for the subsequent files.

In [0]:
from pyspark.sql.functions import col
df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("cloudFiles.schemaLocation", "/Volumes/task11/core/raw_flights_data/_schemas_new")
        .load("/Volumes/task11/core/raw_flights_data")
)

df = df.select(*df.columns[:18])

df = df.dropna()

df.writeStream \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/task11/core/raw_flights_data/_checkpoints_new") \
    .trigger(availableNow=True) \
    .toTable("task11.core.bronze_flights")




In [0]:
%sql
SELECT * FROM task11.core.bronze_flights LIMIT 10;


-- DESCRIBE task11.core.bronze_flights;


year,month,day_of_month,day_of_week,fl_date,origin,origin_city_name,origin_state_nm,dep_time,taxi_out,wheels_off,wheels_on,taxi_in,cancelled,air_time,distance,weather_delay,late_aircraft_delay
